# Eclipse Lab · Sketching the date search

What should someone see after choosing a date? Before connecting the orbital calculation, we can answer that question with two known cases. The interface moves from a date to a result and, for a positive result, to a world map. This notebook is the visual prototype. The integrated notebook supplies the classifier and the actual shadow polygons.

In [ ]:
%pip install streamlit folium

Run from a writable folder in Jupyter. The application looks for an optional photograph named `background.jpg` in `eclipse_dashboard/assets/`.

In [ ]:
import os

os.makedirs("eclipse_dashboard/assets", exist_ok=True)

## Following a date through the interface

Streamlit reruns the script after each interaction. The selected date, the outcome and the current screen therefore belong in session state. April 8, 2024 opens the map, April 9 gives a negative result, and any other date gives an unknown result. Keeping that third case visible prevents an unimplemented search from looking like a physical prediction.

In [ ]:
%%writefile eclipse_dashboard/app.py
import os
import base64
import time
from datetime import date

import streamlit as st
import folium


st.set_page_config(page_title="Eclipse Lab", page_icon="🌘", layout="wide")


def set_background():
    path = os.path.join(os.path.dirname(__file__), "assets", "background.jpg")

    if os.path.exists(path):
        with open(path, "rb") as file:
            image = base64.b64encode(file.read()).decode()
        background = f'url("data:image/jpeg;base64,{image}")'
    else:
        background = "linear-gradient(135deg, #030711, #102643)"

    style = """
    <style>
        .stApp {
            background-image: BACKGROUND;
            background-size: cover;
            background-position: center;
        }
        .block-container {max-width: 1000px; padding-top: 4rem;}
        [data-testid="stHeader"] {background: transparent;}
        .center {text-align: center; margin: 4rem 0 2rem;}
        .title {font-size: 3.5rem; font-weight: bold; color: white;}
        .subtitle {font-size: 1.1rem; color: #bdd0e7;}
        .selected_date {font-size: 3.8rem; font-weight: bold; color: white;}
        .loading {
            position: fixed;
            inset: 0;
            background: #030711;
            z-index: 999999;
            display: flex;
            align-items: center;
            justify-content: center;
            flex-direction: column;
            color: white;
        }
        .circle {
            width: 55px; height: 55px;
            border: 4px solid #223c58;
            border-top: 4px solid #9ecbff;
            border-radius: 50%;
            animation: spin 1s linear infinite;
        }
        @keyframes spin {to {transform: rotate(360deg);}}
    </style>
    """
    st.markdown(style.replace("BACKGROUND", background), unsafe_allow_html=True)


def select_date():
    return st.date_input(
        "Choose a date",
        value=date(2024, 4, 8),
        min_value=date(1, 1, 1),
        max_value=date(3000, 12, 31),
        format="YYYY-MM-DD",
    )


def find_eclipse(selected_date):
    time.sleep(1)

    if selected_date == date(2024, 4, 8):
        return True
    if selected_date == date(2024, 4, 9):
        return False

    return None


def create_map(selected_date):
    map_view = folium.Map(location=[15, 0], zoom_start=2, tiles="CartoDB dark_matter")

    return map_view


if "screen" not in st.session_state:
    st.session_state.screen = "start"

set_background()
st.markdown("**🌘 ECLIPSE LAB**")

if st.session_state.screen == "start":
    st.markdown('''
        <div class="center">
            <div class="title">Explore a solar eclipse</div>
            <div class="subtitle">Choose a date to see the demo result.</div>
        </div>
    ''', unsafe_allow_html=True)

    left, center, right = st.columns([1, 2, 1])

    with center:
        selected_date = select_date()

        if st.button("Find eclipse", use_container_width=True):
            loading = st.empty()
            loading.markdown('''
                <div class="loading">
                    <div class="circle"></div>
                    <h2>Checking the date...</h2>
                </div>
            ''', unsafe_allow_html=True)

            result = find_eclipse(selected_date)
            loading.empty()

            st.session_state.selected_date = selected_date
            st.session_state.result = result
            st.session_state.screen = "result"
            st.rerun()

elif st.session_state.screen == "result":
    selected_date = st.session_state.selected_date
    result = st.session_state.result

    if result is True:
        st.markdown(f'''
            <div class="center">
                <div class="subtitle">Solar eclipse found</div>
                <div class="selected_date">{selected_date.strftime("%d/%m/%Y")}</div>
            </div>
        ''', unsafe_allow_html=True)

        if st.button("View eclipse map", use_container_width=True):
            st.session_state.screen = "map_view"
            st.rerun()

    elif result is False:
        st.markdown("## No solar eclipse on this demo date.")

    else:
        st.info("This date has no demonstration result.")

    if st.button("Choose another date"):
        st.session_state.screen = "start"
        st.rerun()

elif st.session_state.screen == "map_view":
    selected_date = st.session_state.selected_date

    st.title("Eclipse path")
    st.write("Date:", selected_date.strftime("%d/%m/%Y"))
    st.info("Preview map. The shadow layers are calculated in the integrated notebook.")

    map_view = create_map(selected_date)
    st.components.v1.html(map_view.get_root().render(), height=550)

    if st.button("Back to result"):
        st.session_state.screen = "result"
        st.rerun()

    if st.button("Choose another date"):
        st.session_state.screen = "start"
        st.rerun()

## Opening the local map

The notebook starts a Streamlit process and checks for an HTTP response before displaying the link. Keep the kernel open while exploring the demo. Reexecute the application-writing cell after an edit.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display, Markdown

directory = Path.cwd() / "eclipse_dashboard"
file = directory / "app.py"
log = directory / "streamlit.log"
url = "http://127.0.0.1:8501"

if not file.exists():
    print("Run the cell that writes app.py first.")
else:
    if "process" not in globals() or process.poll() is not None:
        with open(log, "w", encoding="utf-8") as output:
            process = subprocess.Popen(
                [sys.executable, "-m", "streamlit", "run", str(file),
                 "--server.address", "127.0.0.1", "--server.port", "8501",
                 "--server.headless", "true"],
                stdout=output,
                stderr=subprocess.STDOUT,
            )

    connected = False

    for attempt in range(20):
        try:
            with urlopen(url, timeout=1) as response:
                connected = response.status == 200
            if connected:
                break
        except Exception:
            time.sleep(0.5)

    if connected:
        print("Dashboard ready.")
        display(Markdown("[**Open Eclipse Lab**](http://127.0.0.1:8501)"))
    else:
        print("Dashboard startup failed. Server log:")
        if log.exists():
            print(log.read_text(encoding="utf-8")[-5000:])

To end the session, set the flag below to `True` and run this cell. Leaving it off keeps the server available after Run All.

In [ ]:
STOP_DASHBOARD = False
if STOP_DASHBOARD and "process" in globals() and process.poll() is None:
    process.terminate()
    process.wait()
    print("Dashboard stopped.")